# Probe V3: Full Pretrained Mistral-7B-Instruct-v0.3 T4 GPU Extraction

**Project:** SQL Injection Security Probe V3 (CWE-89)
**Model:** `mistralai/Mistral-7B-Instruct-v0.3` (32 Layers, Hidden Dim $d = 4,096$)
**Hardware Target:** NVIDIA Tesla T4 GPU (Google Colab / Kaggle)
**Precision:** 4-bit NF4 with Double Quantization (`bitsandbytes`)

---

In [ ]:
# Step 0: Self-Extracting Probe V3 Code & Frozen E2 Datasets
# Unpacks all extraction scripts, AST extractors, manifests, and data splits into the working directory.
import base64, io, zipfile, os

PAYLOAD = """..."""

with zipfile.ZipFile(io.BytesIO(base64.b64decode(PAYLOAD)), "r") as zf:
    zf.extractall(".")

print("SUCCESS: Unpacked all Probe V3 datasets, AST extractors, and scripts!")
if os.path.exists("data/e2_excel/train.jsonl"):
    print(f"Verified data/e2_excel/train.jsonl: {os.path.getsize('data/e2_excel/train.jsonl')} bytes")


## Cell 1: Environment & GPU Hardware Verification
Check Python, PyTorch, CUDA runtime, and available VRAM.

In [ ]:
import sys
import os
import psutil
import torch

print(f"Python Version: {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

assert torch.cuda.is_available(), "CRITICAL: CUDA is not available. Please switch runtime to GPU (T4) in Colab/Kaggle."

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Detected GPU: {gpu_name}")
print(f"Total VRAM: {vram_gb:.2f} GB")

vm = psutil.virtual_memory()
print(f"System RAM: {vm.total / (1024**3):.2f} GB (Available: {vm.available / (1024**3):.2f} GB)")

!df -h / | tail -n 1

## Cell 2: Install Compatible Dependencies
Install Transformers, Accelerate, BitsAndBytes, and dependencies without breaking existing CUDA.

In [ ]:
!pip install -q "transformers>=4.40.0,<4.47.0" "accelerate>=0.29.0" "bitsandbytes>=0.43.0" "safetensors>=0.4.2" "sentencepiece>=0.2.0" "huggingface_hub>=0.22.0" "scikit-learn>=1.4.0"

## Cell 3: Verify bitsandbytes CUDA Compatibility
Confirm that 4-bit NF4 quantization compiles and executes on the active GPU.

In [ ]:
import torch
import bitsandbytes as bnb
print(f"bitsandbytes version: {bnb.__version__}")

# Test 4-bit linear layer on GPU
linear4 = bnb.nn.Linear4bit(128, 128, bias=False, quant_type="nf4", compute_dtype=torch.bfloat16).cuda()
dummy_in = torch.randn(2, 128, dtype=torch.bfloat16).cuda()
dummy_out = linear4(dummy_in)
assert dummy_out.shape == (2, 128), "bitsandbytes 4-bit forward pass failed"
print("SUCCESS: bitsandbytes 4-bit NF4 is fully functional on this GPU.")


## Cell 4: Authenticate with Hugging Face (Optional/Recommended)
Enter a free read token if needed to bypass Hub rate limits.

In [ ]:
from huggingface_hub import login
import os

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass

if hf_token:
    login(token=hf_token)
    print("Authenticated with Hugging Face via token.")
else:
    print("No HF_TOKEN detected. Proceeding with public access.")

## Cell 5: Project Structure & File Integrity Audit
Verify that the frozen Track E2 splits, pilot dataset, and AST sink extractor are present.

In [ ]:
import json

for f in ["data/e2_pilot_20.jsonl", "data/e2_excel/train.jsonl", "data/e2_excel/val.jsonl", "data/e2_excel/test.jsonl", "extraction/ast_sink_extractor.py"]:
    assert os.path.exists(f), f"Missing required project file: {f}"

train_recs = [json.loads(l) for l in open("data/e2_excel/train.jsonl")]
val_recs = [json.loads(l) for l in open("data/e2_excel/val.jsonl")]
test_recs = [json.loads(l) for l in open("data/e2_excel/test.jsonl")]
pilot_recs = [json.loads(l) for l in open("data/e2_pilot_20.jsonl")]

print(f"Frozen Train: {len(train_recs)} samples (Vuln: {sum(1 for r in train_recs if r['label']==1)})")
print(f"Frozen Val:   {len(val_recs)} samples (Vuln: {sum(1 for r in val_recs if r['label']==1)})")
print(f"Frozen Test:  {len(test_recs)} samples (Vuln: {sum(1 for r in test_recs if r['label']==1)})")
print(f"Pilot 20:     {len(pilot_recs)} samples (Vuln: {sum(1 for r in pilot_recs if r['label']==1)}, Safe: {sum(1 for r in pilot_recs if r['label']==0)})")

assert len(train_recs) == 700 and len(val_recs) == 149 and len(test_recs) == 151, "Split count mismatch!"
assert len(pilot_recs) == 20, "Pilot sample count mismatch!"
print("Project data integrity VERIFIED.")

## Cell 6: Phase 1 — Pretrained Model Verification
Execute `verify` mode to load the actual Mistral-7B in 4-bit NF4 and perform a real forward pass with hard assertions.

In [ ]:
!python kaggle_colab_t4_pilot.py --mode verify --data_dir data --output_dir embeddings

## Cell 7: Phase 2 — 20-Sample Pilot Extraction
Extract 4,096-dimensional representations from layers `[4, 8, 12, 16, 20, 24, 28, 32]` for 10 vulnerable and 10 safe samples.
Dual pooling (Mean + AST Sink) calculated simultaneously from the exact same forward pass.

In [ ]:
!python kaggle_colab_t4_pilot.py --mode pilot --data_dir data --output_dir embeddings

## Cell 8: Pilot Gate Audit
Inspect the pilot extraction report and assert that all shapes are `(20, 4096)` with zero NaNs and zero Infs.

In [ ]:
with open("embeddings/pilot_extraction_report.json") as f:
    pilot_report = json.load(f)

print(f"Pilot Status: {pilot_report['status']}")
print(f"Samples: {pilot_report['sample_count']} ({pilot_report['vulnerable_count']} vuln, {pilot_report['safe_count']} safe)")
print(f"Throughput: {pilot_report['throughput_samples_per_sec']:.2f} samples/sec")
print(f"Peak VRAM: {pilot_report['peak_vram_mb']:.2f} MB")

for lyr, info in pilot_report["layer_audit"].items():
    print(f"  {lyr}: Dim={info['dimension']}, Mean Norm={info['mean_norm']:.3f}, Sink Norm={info['sink_norm']:.3f}")

assert pilot_report["status"] == "PILOT_VALIDATION_PASSED", "STOP: Pilot failed! Do not proceed to full extraction."
print("GATE PASSED: Proceeding to full extraction is authorized.")

## Cell 9: Phase 3 — Full Track E2 Extraction (1,000 Samples)
Extract all 1,000 samples across frozen train (700), val (149), and test (151) splits.

In [ ]:
!python kaggle_colab_t4_pilot.py --mode full_e2 --data_dir data --output_dir embeddings --batch_size 8

## Cell 10: Final Manifest Verification & Archive Packaging
Verify checksums and package extracted embeddings into a zip file for transfer back to workstation.

In [ ]:
with open("embeddings/full_e2_extraction_manifest.json") as f:
    full_manifest = json.load(f)

print(f"Full Extraction Status: {full_manifest['status']}")
print(f"Total Samples: {full_manifest['total_samples']}")
print(f"Total Runtime: {full_manifest['total_elapsed_seconds']:.2f}s ({full_manifest['overall_throughput_samples_per_sec']:.2f} samples/sec)")

!zip -r mistral_probe_v3_e2_embeddings.zip embeddings/full_e2/ embeddings/*.json
print("Archive created: mistral_probe_v3_e2_embeddings.zip. Ready for download.")